In [ ]:
import os
from pathlib import Path

import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
import psycopg

PG_HOST = os.getenv("PG_HOST", "localhost")
PG_PORT = int(os.getenv("PG_PORT", "5432"))
PG_DB   = os.getenv("PG_DB", "trading")
PG_USER = os.getenv("PG_USER", "admin")
PG_PWD  = os.getenv("PG_PASSWORD", "admin")

symbol = "BTCUSDT"
interval = "1h"

query = """
SELECT open_time, open, high, low, close, volume
FROM candles
WHERE symbol = %s AND interval = %s
ORDER BY open_time ASC
"""

In [ ]:
PG_HOST = "localhost"
PG_PORT = 5433
PG_DB = "trading"
PG_USER = "admin"
PG_PWD = "admin"

with psycopg.connect(
    host=PG_HOST,
    port=PG_PORT,
    dbname=PG_DB,
    user=PG_USER,
    password=PG_PWD
) as conn:

    df = pd.read_sql(query, conn, params=(symbol, interval))

df.head(), df.shape

C:\Users\fmell\AppData\Local\Temp\ipykernel_8248\623307450.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn, params=(symbol, interval))


(       open_time      open      high       low     close     volume
 0  1735689600000  93576.00  94509.42  93489.03  94401.14  755.99010
 1  1735693200000  94401.13  94408.72  93578.77  93607.74  586.53456
 2  1735696800000  93607.74  94105.12  93594.56  94098.91  276.78045
 3  1735700400000  94098.90  94098.91  93728.22  93838.04  220.99302
 4  1735704000000  93838.04  93838.04  93500.00  93553.91  279.46909,
 (10213, 6))

In [ ]:


# Assurer types numériques
for col in ["open", "high", "low", "close", "volume"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna().reset_index(drop=True)

df.dtypes, df.shape

(open_time      int64
 open         float64
 high         float64
 low          float64
 close        float64
 volume       float64
 dtype: object,
 (10213, 6))

In [ ]:
import pandas as pd
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# ✅ Vérifs indispensables
assert "open" in df.columns, "df n'existe pas ou n'a pas les colonnes attendues"
for col in ["open","high","low","close","volume"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df = df.dropna().reset_index(drop=True)

# ✅ Target (si pas déjà créée)
if "target" not in df.columns:
    df["close_next"] = df["close"].shift(-1)
    df["target"] = (df["close_next"] > df["close"]).astype(int)
    df = df.dropna(subset=["close_next"]).reset_index(drop=True)

FEATURES = ["open","high","low","close","volume"]
X = df[FEATURES]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print("✅ model trained")
print("Train size:", len(X_train), "Test size:", len(X_test))
print("First predictions:", y_pred[:10])
print("Accuracy:", round(acc, 4))

# ✅ Save model.pkl
project_root = Path.cwd().parent  # notebooks/ -> racine projet
out_path = project_root / "models" / "model.pkl"
out_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(model, out_path)
print("✅ saved:", out_path)

In [ ]:
df["close_next"] = df["close"].shift(-1)
df["target"] = (df["close_next"] > df["close"]).astype(int)

# On enlève la dernière ligne (close_next NaN)
df = df.dropna(subset=["close_next"]).reset_index(drop=True)

df["target"].value_counts(), df.shape

(target
 1    5121
 0    5091
 Name: count, dtype: int64,
 (10212, 8))

In [ ]:
df["target"] = (df["close"].shift(-1) > df["close"]).astype(int)

df = df.dropna()

print("Distribution target :")
print(df["target"].value_counts())

Distribution target :
target
1    5121
0    5091
Name: count, dtype: int64


In [ ]:
features = ["open", "high", "low", "close", "volume"]

X = df[features]
y = df["target"]

print(X.head())
print(y.head())

       open      high       low     close     volume
0  93576.00  94509.42  93489.03  94401.14  755.99010
1  94401.13  94408.72  93578.77  93607.74  586.53456
2  93607.74  94105.12  93594.56  94098.91  276.78045
3  94098.90  94098.91  93728.22  93838.04  220.99302
4  93838.04  93838.04  93500.00  93553.91  279.46909
0    0
1    1
2    0
3    0
4    1
Name: target, dtype: int64


In [ ]:
import numpy
import pandas

print(numpy.__version__)
print(pandas.__version__)

2.4.2
3.0.0


In [ ]:
y_pred = model.predict(X_test)
print("First predictions:", y_pred[:10])

NameError: name 'model' is not defined

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import joblib

# 1️⃣ Création des features et target
X = df[["open", "high", "low", "close", "volume"]]
y = df["target"]

# 2️⃣ Train / Test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=False
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

# 3️⃣ Création du modèle
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

# 4️⃣ Entraînement
model.fit(X_train, y_train)
print("✅ Model trained")

# 5️⃣ Prédictions
y_pred = model.predict(X_test)

print("First predictions:", y_pred[:10])

# 6️⃣ Évaluation
acc = accuracy_score(y_test, y_pred)

print("\nAccuracy:", round(acc, 4))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred))

# 7️⃣ Sauvegarde du modèle
joblib.dump(model, "../models/model.pkl")

print("\n✅ Model saved to models/model.pkl")

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

acc = accuracy_score(y_test, y_pred)
print("Accuracy:", round(acc, 4))

print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred, digits=4))

Accuracy: 0.5051

Confusion matrix:
 [[424 604]
 [407 608]]

Classification report:
               precision    recall  f1-score   support

           0     0.5102    0.4125    0.4562      1028
           1     0.5017    0.5990    0.5460      1015

    accuracy                         0.5051      2043
   macro avg     0.5059    0.5057    0.5011      2043
weighted avg     0.5060    0.5051    0.5008      2043



In [ ]:
import joblib

joblib.dump(model, "../models/model.pkl")

print("Model saved")

Model saved


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=False
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 8170
Test size: 2043


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=False
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))






Train size: 8170
Test size: 2043
